# 03 · Gobierno: demostración de REVOKE y acceso de Auditoría
1. Se otorga a `electrocasa_analistas` un privilegio que no le corresponde (SELECT sobre silver),
   se muestra `SHOW GRANTS`, se revoca y se vuelve a mostrar.
2. Se otorga a `electrocasa_auditoria` acceso a `system.access` (audit y lineage).
   Si falla por permisos, la celda muestra el error y la evidencia alternativa es la pestaña
   **Lineage** de Catalog Explorer (captura en el README).

In [ ]:
dbutils.widgets.text("catalog", "electrocasa_dev", "Catálogo del proyecto")
catalogo = dbutils.widgets.get("catalog")
print("Catálogo:", catalogo)

In [ ]:
sql = """
-- ANTES: privilegio indebido
GRANT USE SCHEMA, SELECT ON SCHEMA ${electrocasa.catalog}.silver TO `electrocasa_analistas`;
SHOW GRANTS `electrocasa_analistas` ON SCHEMA ${electrocasa.catalog}.silver;
""".replace("${electrocasa.catalog}", catalogo)
resultado = None
for sentencia in [s.strip() for s in sql.split(";")]:
    if sentencia and not all(l.strip().startswith("--") for l in sentencia.splitlines() if l.strip()):
        resultado = spark.sql(sentencia)
if resultado is not None and len(resultado.columns) > 0:
    display(resultado)

In [ ]:
sql = """
-- REVOKE
REVOKE USE SCHEMA, SELECT ON SCHEMA ${electrocasa.catalog}.silver FROM `electrocasa_analistas`;
""".replace("${electrocasa.catalog}", catalogo)
resultado = None
for sentencia in [s.strip() for s in sql.split(";")]:
    if sentencia and not all(l.strip().startswith("--") for l in sentencia.splitlines() if l.strip()):
        resultado = spark.sql(sentencia)
if resultado is not None and len(resultado.columns) > 0:
    display(resultado)

In [ ]:
sql = """
-- DESPUÉS: analistas ya no tiene privilegios sobre silver (resultado vacío)
SHOW GRANTS `electrocasa_analistas` ON SCHEMA ${electrocasa.catalog}.silver;
""".replace("${electrocasa.catalog}", catalogo)
resultado = None
for sentencia in [s.strip() for s in sql.split(";")]:
    if sentencia and not all(l.strip().startswith("--") for l in sentencia.splitlines() if l.strip()):
        resultado = spark.sql(sentencia)
if resultado is not None and len(resultado.columns) > 0:
    display(resultado)

In [ ]:
sql = """
-- Analistas conserva su acceso a gold
SHOW GRANTS `electrocasa_analistas` ON SCHEMA ${electrocasa.catalog}.gold;
""".replace("${electrocasa.catalog}", catalogo)
resultado = None
for sentencia in [s.strip() for s in sql.split(";")]:
    if sentencia and not all(l.strip().startswith("--") for l in sentencia.splitlines() if l.strip()):
        resultado = spark.sql(sentencia)
if resultado is not None and len(resultado.columns) > 0:
    display(resultado)

## Acceso de Auditoría a system.access
Requiere que quien lo ejecuta tenga `MANAGE` (o sea dueño) sobre el catálogo `system` / los schemas habilitados.

In [ ]:
sentencias = [
    "GRANT USE CATALOG ON CATALOG system TO `electrocasa_auditoria`",
    "GRANT USE SCHEMA ON SCHEMA system.access TO `electrocasa_auditoria`",
    "GRANT SELECT ON TABLE system.access.audit TO `electrocasa_auditoria`",
    "GRANT SELECT ON TABLE system.access.table_lineage TO `electrocasa_auditoria`",
    "GRANT SELECT ON TABLE system.access.column_lineage TO `electrocasa_auditoria`",
]
for s in sentencias:
    try:
        spark.sql(s)
        print("OK   ", s)
    except Exception as e:
        print("ERROR", s, "->", str(e).splitlines()[0])

In [ ]:
sql = """
-- Evidencia de lineage del proyecto (si system.access está disponible)
SELECT source_table_full_name, target_table_full_name, entity_type, max(event_time) AS ultimo_evento
FROM system.access.table_lineage
WHERE target_table_catalog = '${electrocasa.catalog}'
GROUP BY ALL
ORDER BY target_table_full_name;
""".replace("${electrocasa.catalog}", catalogo)
resultado = None
for sentencia in [s.strip() for s in sql.split(";")]:
    if sentencia and not all(l.strip().startswith("--") for l in sentencia.splitlines() if l.strip()):
        resultado = spark.sql(sentencia)
if resultado is not None and len(resultado.columns) > 0:
    display(resultado)

In [ ]:
sql = """
-- Evidencia de auditoría: quién otorgó o revocó permisos sobre el catálogo del proyecto
SELECT event_time, user_identity.email AS usuario, action_name, request_params
FROM system.access.audit
WHERE service_name = 'unityCatalog'
  AND action_name IN ('updatePermissions', 'grantPermission', 'revokePermission')
  AND event_date >= current_date() - INTERVAL 7 DAYS
ORDER BY event_time DESC
LIMIT 50;
""".replace("${electrocasa.catalog}", catalogo)
resultado = None
for sentencia in [s.strip() for s in sql.split(";")]:
    if sentencia and not all(l.strip().startswith("--") for l in sentencia.splitlines() if l.strip()):
        resultado = spark.sql(sentencia)
if resultado is not None and len(resultado.columns) > 0:
    display(resultado)